# CardioVUS-KCNH2 — XGBoost modeling

**Purpose:** predict the experimental KCNH2 surface-trafficking score for individual missense variants.

Four models are evaluated with the same residue-grouped folds:

1. **Model 0 — Constant median baseline**
2. **Model 1 — XGBoost with biochemical and substitution features**
3. **Model 2 — XGBoost with biochemical and UniProt domain features**
4. **Model 3 — XGBoost with biochemical, domain, and ESM-2 WT contextual embeddings**

The target is a molecular surface-trafficking measurement. It is not a direct prediction of pathogenicity, QT interval, arrhythmia risk, or ACMG/AMP classification.

## Recommended execution strategy

**Run 1:** run Models 0–2 and leave ESM-2 disabled.

**Run 2:** attach the output files from Run 1 plus the offline ESM-2 bundle, disable Models 0–2, and enable only Model 3. The notebook will merge the previous OOF metrics and predictions with the new ESM-2 results.

## 1. Central configuration — purpose: control paths, folds, models, XGBoost, and ESM-2 from one class

The dataset already contains `fold_3` and `fold_5`, generated with `GroupKFold(group=position)`. The MVP uses three folds to reduce runtime while preserving residue-level separation.

In [ ]:
from __future__ import annotations

import gc
import json
import platform
import random
import re
import shutil
import time
import warnings
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import torch
import transformers
import xgboost as xgb
from IPython.display import display
from packaging.version import Version
from scipy import stats
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from transformers import AutoModel, AutoTokenizer

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)


@dataclass(frozen=True)
class CFG:
    # ---------------------------------------------------------
    # Experiment switches
    # ---------------------------------------------------------
    RUN_BASELINE: bool = True
    RUN_XGB_BIOCHEMICAL: bool = True
    RUN_XGB_DOMAINS: bool = True
    RUN_XGB_ESM2: bool = False

    LOAD_PREVIOUS_RESULTS: bool = True

    # ---------------------------------------------------------
    # Reproducibility and CV
    # ---------------------------------------------------------
    SEED: int = 42
    N_SPLITS: int = 3
    FOLD_COLUMN: str = "fold_3"
    TARGET_COLUMN: str = "score_numeric"
    GROUP_COLUMN: str = "position"

    # ---------------------------------------------------------
    # Kaggle paths
    # ---------------------------------------------------------
    KAGGLE_INPUT_DIR: Path = Path("/kaggle/input")
    WORKING_DIR: Path = Path("/kaggle/working")
    OUTPUT_DIR: Path = WORKING_DIR / "cardiovus_outputs"

    METRICS_DIR: Path = OUTPUT_DIR / "metrics"
    PREDICTIONS_DIR: Path = OUTPUT_DIR / "predictions"
    MODELS_DIR: Path = OUTPUT_DIR / "models"
    FIGURES_DIR: Path = OUTPUT_DIR / "figures"
    EMBEDDINGS_DIR: Path = OUTPUT_DIR / "embeddings"

    # Optional explicit paths. Leave as None for auto-discovery.
    MODELING_DATASET_PATH: str | None = None
    FEATURE_SCHEMA_PATH: str | None = None
    REFERENCE_FASTA_PATH: str | None = None
    ESM2_LOCAL_MODEL_PATH: str | None = None

    # ---------------------------------------------------------
    # XGBoost
    # Same hyperparameters are used for Models 1–3 so the
    # comparison mainly reflects feature sets.
    # ---------------------------------------------------------
    USE_XGB_GPU: bool = True
    XGB_N_ESTIMATORS: int = 1600
    XGB_EARLY_STOPPING_ROUNDS: int = 80
    XGB_LEARNING_RATE: float = 0.035
    XGB_MAX_DEPTH: int = 6
    XGB_MIN_CHILD_WEIGHT: float = 5.0
    XGB_SUBSAMPLE: float = 0.85
    XGB_COLSAMPLE_BYTREE: float = 0.80
    XGB_REG_ALPHA: float = 0.10
    XGB_REG_LAMBDA: float = 5.0
    XGB_MAX_BIN: int = 256
    XGB_N_JOBS: int = 2

    # ---------------------------------------------------------
    # ESM-2
    # ---------------------------------------------------------
    ESM2_MODEL_NAME: str = "facebook/esm2_t6_8M_UR50D"
    ESM2_ALLOW_INTERNET_DOWNLOAD: bool = False
    ESM2_WINDOW_SIZE: int = 900
    ESM2_STRIDE: int = 600
    ESM2_USE_FP16: bool = True

    # ---------------------------------------------------------
    # Output and analysis
    # ---------------------------------------------------------
    SAVE_FOLD_MODELS: bool = True
    SAVE_FEATURE_IMPORTANCE: bool = True
    TOP_IMPORTANCE_FEATURES: int = 40
    FUNCTIONAL_THRESHOLDS: tuple[float, float, float] = (
        35.0,
        58.0,
        153.0,
    )


cfg = CFG()

for directory in [
    cfg.OUTPUT_DIR,
    cfg.METRICS_DIR,
    cfg.PREDICTIONS_DIR,
    cfg.MODELS_DIR,
    cfg.FIGURES_DIR,
    cfg.EMBEDDINGS_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

random.seed(cfg.SEED)
np.random.seed(cfg.SEED)
torch.manual_seed(cfg.SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(cfg.SEED)

print("Python:", platform.python_version())
print("pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("SciPy:", scipy.__version__)
print("XGBoost:", xgb.__version__)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print()
print("Run configuration:")
print(
    json.dumps(
        {
            key: str(value) if isinstance(value, Path) else value
            for key, value in asdict(cfg).items()
        },
        indent=2,
    )
)

## 2. Input discovery — purpose: find the modeling dataset, schema, FASTA, ESM-2 bundle, and prior results

The notebook searches recursively under `/kaggle/input`, so Kaggle's generated dataset slugs do not need to be hard-coded.

In [ ]:
def find_unique_file(
    filename: str,
    explicit_path: str | None = None,
    required: bool = True,
) -> Path | None:
    if explicit_path is not None:
        path = Path(explicit_path)

        if not path.exists():
            raise FileNotFoundError(
                f"Explicit path does not exist: {path}"
            )

        return path

    matches = sorted(cfg.KAGGLE_INPUT_DIR.rglob(filename))

    if not matches:
        if required:
            raise FileNotFoundError(
                f"Could not find {filename} under "
                f"{cfg.KAGGLE_INPUT_DIR}"
            )

        return None

    if len(matches) > 1:
        print(
            f"[WARNING] Multiple files named {filename} "
            "were found. The first one will be used:"
        )

        for match in matches:
            print(" -", match)

    return matches[0]


def find_esm2_model_directory() -> Path | None:
    if cfg.ESM2_LOCAL_MODEL_PATH is not None:
        path = Path(cfg.ESM2_LOCAL_MODEL_PATH)

        if not path.exists():
            raise FileNotFoundError(
                f"ESM-2 path does not exist: {path}"
            )

        return path

    candidates: list[Path] = []

    for config_path in cfg.KAGGLE_INPUT_DIR.rglob("config.json"):
        parent = config_path.parent
        files = {path.name for path in parent.iterdir()}

        has_tokenizer = (
            "tokenizer_config.json" in files
            or "vocab.txt" in files
        )

        has_weights = (
            "model.safetensors" in files
            or "pytorch_model.bin" in files
        )

        if has_tokenizer and has_weights:
            candidates.append(parent)

    esm_candidates = [
        path
        for path in candidates
        if "esm" in str(path).lower()
    ]

    selected_candidates = (
        esm_candidates if esm_candidates else candidates
    )

    if not selected_candidates:
        return None

    selected_candidates = sorted(set(selected_candidates))

    if len(selected_candidates) > 1:
        print(
            "[WARNING] Multiple local model directories "
            "were detected:"
        )

        for candidate in selected_candidates:
            print(" -", candidate)

    return selected_candidates[0]


dataset_path = find_unique_file(
    "kcnh2_modeling_features.parquet",
    cfg.MODELING_DATASET_PATH,
)

schema_path = find_unique_file(
    "kcnh2_modeling_feature_schema.json",
    cfg.FEATURE_SCHEMA_PATH,
)

fasta_path = find_unique_file(
    "NP_000229.1.fasta",
    cfg.REFERENCE_FASTA_PATH,
    required=cfg.RUN_XGB_ESM2,
)

esm2_model_directory = (
    find_esm2_model_directory()
    if cfg.RUN_XGB_ESM2
    else None
)

print("Modeling dataset:", dataset_path)
print("Feature schema:", schema_path)
print("Reference FASTA:", fasta_path)
print("Local ESM-2 model:", esm2_model_directory)

## 3. Load and validate the modeling dataset — purpose: guarantee identical folds and prevent target leakage

Every substitution affecting the same residue must have the same fold assignment. The notebook uses the precomputed fold column instead of generating a new split during every run.

In [ ]:
with schema_path.open("r", encoding="utf-8") as file:
    feature_schema = json.load(file)

df = pd.read_parquet(dataset_path)

biochemical_features = feature_schema[
    "model_1"
]["feature_columns"]

domain_features = feature_schema[
    "domain_feature_columns"
]

biochemical_domain_features = feature_schema[
    "model_2"
]["feature_columns"]

required_columns = {
    "variant_id",
    "wt_aa1",
    "mut_aa1",
    "position",
    cfg.TARGET_COLUMN,
    cfg.FOLD_COLUMN,
}

missing_required = required_columns - set(df.columns)

if missing_required:
    raise ValueError(
        "Missing required dataset columns: "
        f"{sorted(missing_required)}"
    )

if len(df) != 20_683:
    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )

if not df["variant_id"].is_unique:
    raise ValueError("variant_id must be unique.")

if df[cfg.TARGET_COLUMN].isna().any():
    raise ValueError("The target contains missing values.")

all_model_features = list(
    dict.fromkeys(biochemical_domain_features)
)

missing_features = set(all_model_features) - set(df.columns)

if missing_features:
    raise ValueError(
        "Feature schema references missing columns: "
        f"{sorted(missing_features)}"
    )

if df[all_model_features].isna().any().any():
    raise ValueError(
        "Predictive features contain missing values."
    )

fold_values = sorted(
    df[cfg.FOLD_COLUMN].unique().tolist()
)

expected_folds = list(range(cfg.N_SPLITS))

if fold_values != expected_folds:
    raise ValueError(
        f"Unexpected fold values: {fold_values}. "
        f"Expected {expected_folds}."
    )

folds_per_position = (
    df.groupby(cfg.GROUP_COLUMN)[
        cfg.FOLD_COLUMN
    ].nunique()
)

if not folds_per_position.eq(1).all():
    raise ValueError(
        "Position leakage detected across folds."
    )

prohibited_predictors = {
    "score",
    "se",
    "LLR",
    "LLR_ci_lower",
    "LLR_ci_upper",
    "LLR_evidence_strength",
}

leakage_found = prohibited_predictors & set(all_model_features)

if leakage_found:
    raise ValueError(
        "Experimental leakage columns found: "
        f"{sorted(leakage_found)}"
    )

df["position"] = df["position"].astype("int16")

y = df[cfg.TARGET_COLUMN].to_numpy(
    dtype=np.float32
)

fold_assignments = df[
    cfg.FOLD_COLUMN
].to_numpy(dtype=np.int8)

print("Dataset shape:", df.shape)
print("Unique positions:", df["position"].nunique())
print("Biochemical features:", len(biochemical_features))
print("Domain features:", len(domain_features))
print(
    "Biochemical + domain features:",
    len(biochemical_domain_features),
)
print()
print("Fold distribution:")
print(
    df[cfg.FOLD_COLUMN]
    .value_counts()
    .sort_index()
)

## 4. Evaluation functions — purpose: calculate the same honest metrics for every model

- **Spearman:** primary ranking metric.
- **Pearson:** linear association.
- **MAE:** average absolute error.
- **RMSE:** penalizes large errors more strongly.
- **R²:** proportion of variance explained.

In [ ]:
def safe_spearman(
    y_true: np.ndarray,
    y_pred: np.ndarray,
) -> float:
    if np.std(y_true) == 0 or np.std(y_pred) == 0:
        return float("nan")

    return float(
        stats.spearmanr(
            y_true,
            y_pred,
        ).statistic
    )


def safe_pearson(
    y_true: np.ndarray,
    y_pred: np.ndarray,
) -> float:
    if np.std(y_true) == 0 or np.std(y_pred) == 0:
        return float("nan")

    return float(
        stats.pearsonr(
            y_true,
            y_pred,
        ).statistic
    )


def calculate_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
) -> dict[str, float]:
    return {
        "spearman": safe_spearman(y_true, y_pred),
        "pearson": safe_pearson(y_true, y_pred),
        "mae": float(
            mean_absolute_error(y_true, y_pred)
        ),
        "rmse": float(
            np.sqrt(
                mean_squared_error(y_true, y_pred)
            )
        ),
        "r2": float(
            r2_score(y_true, y_pred)
        ),
    }


def assign_functional_category(
    values: pd.Series | np.ndarray,
) -> np.ndarray:
    severe, reduced, upper = (
        cfg.FUNCTIONAL_THRESHOLDS
    )

    values = np.asarray(values)

    return np.select(
        [
            values < severe,
            (values >= severe) & (values < reduced),
            (values >= reduced) & (values <= upper),
            values > upper,
        ],
        [
            "severe_loss_lt_35",
            "reduced_35_to_58",
            "wt_like_58_to_153",
            "above_wt_gt_153",
        ],
        default="unknown",
    )


def build_oof_frame(
    model_name: str,
    predictions: np.ndarray,
) -> pd.DataFrame:
    output = df[
        [
            "variant_id",
            "position",
            cfg.FOLD_COLUMN,
            cfg.TARGET_COLUMN,
        ]
    ].copy()

    output = output.rename(
        columns={
            cfg.FOLD_COLUMN: "fold",
            cfg.TARGET_COLUMN: "y_true",
        }
    )

    output["y_pred"] = predictions
    output["residual"] = (
        output["y_true"] - output["y_pred"]
    )
    output["absolute_error"] = (
        output["residual"].abs()
    )
    output["functional_category"] = (
        assign_functional_category(
            output["y_true"]
        )
    )
    output["model"] = model_name

    return output


current_metric_rows: list[dict[str, Any]] = []
current_oof_frames: list[pd.DataFrame] = []
current_importance_frames: list[pd.DataFrame] = []

## 5. Previous-run outputs — purpose: compare the second ESM-2 run with Models 0–2 without retraining them

When the output files from Run 1 are attached as a Kaggle input, this cell loads them automatically.

In [ ]:
def load_previous_results() -> tuple[
    pd.DataFrame,
    pd.DataFrame,
]:
    empty_metrics = pd.DataFrame()
    empty_oof = pd.DataFrame()

    if not cfg.LOAD_PREVIOUS_RESULTS:
        return empty_metrics, empty_oof

    metric_candidates = sorted(
        cfg.KAGGLE_INPUT_DIR.rglob(
            "model_metrics.csv"
        )
    )

    oof_candidates = sorted(
        cfg.KAGGLE_INPUT_DIR.rglob(
            "oof_predictions.parquet"
        )
    )

    previous_metrics = empty_metrics
    previous_oof = empty_oof

    if metric_candidates:
        previous_metrics = pd.read_csv(
            metric_candidates[0]
        )
        print(
            "Loaded previous metrics:",
            metric_candidates[0],
        )

    if oof_candidates:
        previous_oof = pd.read_parquet(
            oof_candidates[0]
        )
        print(
            "Loaded previous OOF:",
            oof_candidates[0],
        )

    return previous_metrics, previous_oof


previous_metrics_df, previous_oof_df = (
    load_previous_results()
)

if not previous_metrics_df.empty:
    display(
        previous_metrics_df.loc[
            previous_metrics_df[
                "fold"
            ].astype(str).eq("OOF")
        ]
    )

## 6. Model 0 — purpose: establish the minimum performance any useful model must exceed

For each validation fold, the baseline predicts the median score of the corresponding training folds. It uses no biological features.

In [ ]:
def run_constant_median_baseline() -> tuple[
    pd.DataFrame,
    pd.DataFrame,
]:
    model_name = "model_0_median_baseline"
    predictions = np.full(
        len(df),
        np.nan,
        dtype=np.float32,
    )

    metric_rows: list[dict[str, Any]] = []
    start_time = time.time()

    for fold in range(cfg.N_SPLITS):
        train_mask = fold_assignments != fold
        validation_mask = fold_assignments == fold

        training_median = float(
            np.median(y[train_mask])
        )

        predictions[
            validation_mask
        ] = training_median

        fold_metrics = calculate_metrics(
            y[validation_mask],
            predictions[validation_mask],
        )

        metric_rows.append(
            {
                "model": model_name,
                "fold": str(fold),
                "n_validation": int(
                    validation_mask.sum()
                ),
                "n_features": 0,
                "best_iteration": np.nan,
                "runtime_seconds": (
                    time.time() - start_time
                ),
                **fold_metrics,
            }
        )

    if np.isnan(predictions).any():
        raise ValueError(
            "Baseline OOF predictions are incomplete."
        )

    oof_metrics = calculate_metrics(
        y,
        predictions,
    )

    metric_rows.append(
        {
            "model": model_name,
            "fold": "OOF",
            "n_validation": len(df),
            "n_features": 0,
            "best_iteration": np.nan,
            "runtime_seconds": (
                time.time() - start_time
            ),
            **oof_metrics,
        }
    )

    return (
        pd.DataFrame(metric_rows),
        build_oof_frame(
            model_name,
            predictions,
        ),
    )


if cfg.RUN_BASELINE:
    baseline_metrics, baseline_oof = (
        run_constant_median_baseline()
    )

    current_metric_rows.extend(
        baseline_metrics.to_dict(
            orient="records"
        )
    )

    current_oof_frames.append(
        baseline_oof
    )

    display(
        baseline_metrics.loc[
            baseline_metrics["fold"].eq(
                "OOF"
            )
        ]
    )

## 7. Shared XGBoost training function — purpose: ensure Models 1–3 differ only in their feature sets

The notebook first attempts GPU XGBoost. If the installed XGBoost/CUDA combination fails, it automatically retries on CPU using histogram trees.

In [ ]:
def build_xgb_parameters(
    use_gpu: bool,
) -> dict[str, Any]:
    parameters: dict[str, Any] = {
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "n_estimators": cfg.XGB_N_ESTIMATORS,
        "learning_rate": cfg.XGB_LEARNING_RATE,
        "max_depth": cfg.XGB_MAX_DEPTH,
        "min_child_weight": cfg.XGB_MIN_CHILD_WEIGHT,
        "subsample": cfg.XGB_SUBSAMPLE,
        "colsample_bytree": cfg.XGB_COLSAMPLE_BYTREE,
        "reg_alpha": cfg.XGB_REG_ALPHA,
        "reg_lambda": cfg.XGB_REG_LAMBDA,
        "max_bin": cfg.XGB_MAX_BIN,
        "random_state": cfg.SEED,
        "n_jobs": cfg.XGB_N_JOBS,
        "tree_method": "hist",
        "early_stopping_rounds": (
            cfg.XGB_EARLY_STOPPING_ROUNDS
        ),
    }

    xgb_version = Version(xgb.__version__)

    if use_gpu and torch.cuda.is_available():
        if xgb_version >= Version("2.0.0"):
            parameters["device"] = "cuda"
        else:
            parameters["tree_method"] = "gpu_hist"
            parameters["predictor"] = "gpu_predictor"
    elif xgb_version >= Version("2.0.0"):
        parameters["device"] = "cpu"

    return parameters


def fit_one_xgb_fold(
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_validation: np.ndarray,
    y_validation: np.ndarray,
) -> xgb.XGBRegressor:
    use_gpu = (
        cfg.USE_XGB_GPU
        and torch.cuda.is_available()
    )

    parameters = build_xgb_parameters(
        use_gpu=use_gpu
    )

    model = xgb.XGBRegressor(
        **parameters
    )

    try:
        model.fit(
            X_train,
            y_train,
            eval_set=[
                (
                    X_validation,
                    y_validation,
                )
            ],
            verbose=False,
        )
        return model

    except xgb.core.XGBoostError as error:
        if not use_gpu:
            raise

        print(
            "[WARNING] GPU XGBoost failed. "
            "Retrying this fold on CPU."
        )
        print(error)

        cpu_parameters = build_xgb_parameters(
            use_gpu=False
        )

        model = xgb.XGBRegressor(
            **cpu_parameters
        )

        model.fit(
            X_train,
            y_train,
            eval_set=[
                (
                    X_validation,
                    y_validation,
                )
            ],
            verbose=False,
        )

        return model


def extract_gain_importance(
    model: xgb.XGBRegressor,
    feature_names: list[str],
    model_name: str,
    fold: int,
) -> pd.DataFrame:
    booster_scores = (
        model.get_booster().get_score(
            importance_type="gain"
        )
    )

    rows = []

    for raw_feature, importance in (
        booster_scores.items()
    ):
        match = re.fullmatch(
            r"f(\d+)",
            raw_feature,
        )

        if match:
            feature_index = int(
                match.group(1)
            )
            feature_name = (
                feature_names[
                    feature_index
                ]
            )
        else:
            feature_name = raw_feature

        rows.append(
            {
                "model": model_name,
                "fold": fold,
                "feature": feature_name,
                "gain": float(importance),
            }
        )

    return pd.DataFrame(rows)


def run_xgboost_cv(
    X: np.ndarray,
    feature_names: list[str],
    model_name: str,
) -> tuple[
    pd.DataFrame,
    pd.DataFrame,
    pd.DataFrame,
]:
    if X.shape[0] != len(df):
        raise ValueError(
            "X row count does not match the dataset."
        )

    if X.shape[1] != len(feature_names):
        raise ValueError(
            "X column count does not match feature_names."
        )

    if not np.isfinite(X).all():
        raise ValueError(
            f"{model_name} contains NaN or infinite values."
        )

    predictions = np.full(
        len(df),
        np.nan,
        dtype=np.float32,
    )

    metric_rows: list[dict[str, Any]] = []
    importance_frames: list[pd.DataFrame] = []
    model_start_time = time.time()

    for fold in range(cfg.N_SPLITS):
        fold_start_time = time.time()

        train_mask = fold_assignments != fold
        validation_mask = fold_assignments == fold

        X_train = np.ascontiguousarray(
            X[train_mask],
            dtype=np.float32,
        )
        X_validation = np.ascontiguousarray(
            X[validation_mask],
            dtype=np.float32,
        )

        y_train = y[train_mask]
        y_validation = y[validation_mask]

        print()
        print("=" * 76)
        print(
            f"{model_name} — fold {fold + 1}/"
            f"{cfg.N_SPLITS}"
        )
        print("=" * 76)
        print(
            "Train:",
            X_train.shape,
            "| Validation:",
            X_validation.shape,
        )

        model = fit_one_xgb_fold(
            X_train=X_train,
            y_train=y_train,
            X_validation=X_validation,
            y_validation=y_validation,
        )

        fold_predictions = model.predict(
            X_validation
        ).astype(np.float32)

        predictions[
            validation_mask
        ] = fold_predictions

        fold_metrics = calculate_metrics(
            y_validation,
            fold_predictions,
        )

        best_iteration = getattr(
            model,
            "best_iteration",
            None,
        )

        metric_rows.append(
            {
                "model": model_name,
                "fold": str(fold),
                "n_validation": int(
                    validation_mask.sum()
                ),
                "n_features": int(X.shape[1]),
                "best_iteration": (
                    int(best_iteration)
                    if best_iteration
                    is not None
                    else np.nan
                ),
                "runtime_seconds": (
                    time.time()
                    - fold_start_time
                ),
                **fold_metrics,
            }
        )

        print(
            json.dumps(
                fold_metrics,
                indent=2,
            )
        )

        if cfg.SAVE_FOLD_MODELS:
            model_path = (
                cfg.MODELS_DIR
                / f"{model_name}_fold_{fold}.json"
            )
            model.save_model(str(model_path))

        if cfg.SAVE_FEATURE_IMPORTANCE:
            importance_frames.append(
                extract_gain_importance(
                    model=model,
                    feature_names=feature_names,
                    model_name=model_name,
                    fold=fold,
                )
            )

        del (
            model,
            X_train,
            X_validation,
            y_train,
            y_validation,
        )
        gc.collect()

    if np.isnan(predictions).any():
        raise ValueError(
            f"{model_name} OOF predictions are incomplete."
        )

    oof_metrics = calculate_metrics(
        y,
        predictions,
    )

    metric_rows.append(
        {
            "model": model_name,
            "fold": "OOF",
            "n_validation": len(df),
            "n_features": int(X.shape[1]),
            "best_iteration": np.nan,
            "runtime_seconds": (
                time.time()
                - model_start_time
            ),
            **oof_metrics,
        }
    )

    importance_df = (
        pd.concat(
            importance_frames,
            ignore_index=True,
        )
        if importance_frames
        else pd.DataFrame()
    )

    return (
        pd.DataFrame(metric_rows),
        build_oof_frame(
            model_name,
            predictions,
        ),
        importance_df,
    )

## 8. Model 1 — purpose: measure how much trafficking can be predicted from the substitution and simple biochemical context

In [ ]:
if cfg.RUN_XGB_BIOCHEMICAL:
    X_biochemical = (
        df[biochemical_features]
        .to_numpy(dtype=np.float32)
    )

    (
        biochemical_metrics,
        biochemical_oof,
        biochemical_importance,
    ) = run_xgboost_cv(
        X=X_biochemical,
        feature_names=biochemical_features,
        model_name=(
            "model_1_xgb_biochemical"
        ),
    )

    current_metric_rows.extend(
        biochemical_metrics.to_dict(
            orient="records"
        )
    )
    current_oof_frames.append(
        biochemical_oof
    )

    if not biochemical_importance.empty:
        current_importance_frames.append(
            biochemical_importance
        )

    display(
        biochemical_metrics.loc[
            biochemical_metrics[
                "fold"
            ].eq("OOF")
        ]
    )

    del X_biochemical
    gc.collect()

## 9. Model 2 — purpose: quantify the additional predictive value of KCNH2 domain and topology annotations

In [ ]:
if cfg.RUN_XGB_DOMAINS:
    X_domains = (
        df[
            biochemical_domain_features
        ]
        .to_numpy(dtype=np.float32)
    )

    (
        domain_metrics,
        domain_oof,
        domain_importance,
    ) = run_xgboost_cv(
        X=X_domains,
        feature_names=(
            biochemical_domain_features
        ),
        model_name=(
            "model_2_xgb_biochemical_domains"
        ),
    )

    current_metric_rows.extend(
        domain_metrics.to_dict(
            orient="records"
        )
    )
    current_oof_frames.append(
        domain_oof
    )

    if not domain_importance.empty:
        current_importance_frames.append(
            domain_importance
        )

    display(
        domain_metrics.loc[
            domain_metrics[
                "fold"
            ].eq("OOF")
        ]
    )

    del X_domains
    gc.collect()

## 10. ESM-2 reference embedding — purpose: represent the evolutionary and sequence context of every KCNH2 residue

KCNH2 is processed in overlapping windows because its sequence is longer than the selected checkpoint's context window. Overlapping residue embeddings are averaged.

In [ ]:
def read_single_fasta(
    path: Path,
) -> tuple[str, str]:
    header: str | None = None
    sequence_parts: list[str] = []

    with path.open(
        "r",
        encoding="utf-8",
    ) as file:
        for line in file:
            line = line.strip()

            if not line:
                continue

            if line.startswith(">"):
                if header is not None:
                    raise ValueError(
                        "The FASTA contains more than "
                        "one sequence."
                    )

                header = line[1:].split()[0]
            else:
                sequence_parts.append(line)

    if header is None:
        raise ValueError(
            "No FASTA header was found."
        )

    sequence = "".join(
        sequence_parts
    ).upper()

    return header, sequence


def build_window_starts(
    sequence_length: int,
    window_size: int,
    stride: int,
) -> list[int]:
    if window_size <= 0 or stride <= 0:
        raise ValueError(
            "Window size and stride must be positive."
        )

    if sequence_length <= window_size:
        return [0]

    starts = list(
        range(
            0,
            sequence_length
            - window_size
            + 1,
            stride,
        )
    )

    final_start = (
        sequence_length - window_size
    )

    if starts[-1] != final_start:
        starts.append(final_start)

    return sorted(set(starts))


def load_esm2_model() -> tuple[
    Any,
    Any,
    str,
]:
    if esm2_model_directory is not None:
        model_source = str(
            esm2_model_directory
        )
        local_files_only = True
    elif cfg.ESM2_ALLOW_INTERNET_DOWNLOAD:
        model_source = (
            cfg.ESM2_MODEL_NAME
        )
        local_files_only = False
    else:
        raise FileNotFoundError(
            "RUN_XGB_ESM2=True, but no offline ESM-2 "
            "bundle was found. Upload a directory containing "
            "config.json, tokenizer files, and model weights, "
            "or set ESM2_ALLOW_INTERNET_DOWNLOAD=True."
        )

    tokenizer = AutoTokenizer.from_pretrained(
        model_source,
        local_files_only=local_files_only,
    )

    model = AutoModel.from_pretrained(
        model_source,
        local_files_only=local_files_only,
    )

    return (
        tokenizer,
        model,
        model_source,
    )


def extract_reference_embeddings(
    fasta_file: Path,
) -> tuple[
    np.ndarray,
    dict[str, Any],
]:
    embedding_path = (
        cfg.EMBEDDINGS_DIR
        / "esm2_wt_position_embeddings.npy"
    )

    metadata_path = (
        cfg.EMBEDDINGS_DIR
        / "esm2_wt_embedding_metadata.json"
    )

    if (
        embedding_path.exists()
        and metadata_path.exists()
    ):
        embeddings = np.load(
            embedding_path
        )

        with metadata_path.open(
            "r",
            encoding="utf-8",
        ) as file:
            metadata = json.load(file)

        print(
            "Loaded cached ESM-2 embeddings:",
            embedding_path,
        )

        return embeddings, metadata

    reference_id, sequence = (
        read_single_fasta(
            fasta_file
        )
    )

    if reference_id != "NP_000229.1":
        raise ValueError(
            f"Unexpected FASTA ID: {reference_id}"
        )

    if len(sequence) != 1159:
        raise ValueError(
            f"Unexpected sequence length: "
            f"{len(sequence)}"
        )

    tokenizer, model, model_source = (
        load_esm2_model()
    )

    device = torch.device(
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )

    model = model.to(device)
    model.eval()

    hidden_size = int(
        model.config.hidden_size
    )

    embedding_sum = np.zeros(
        (
            len(sequence),
            hidden_size,
        ),
        dtype=np.float32,
    )

    embedding_count = np.zeros(
        len(sequence),
        dtype=np.int16,
    )

    window_starts = build_window_starts(
        sequence_length=len(sequence),
        window_size=(
            cfg.ESM2_WINDOW_SIZE
        ),
        stride=cfg.ESM2_STRIDE,
    )

    print(
        "ESM-2 windows:",
        [
            (
                start + 1,
                min(
                    start
                    + cfg.ESM2_WINDOW_SIZE,
                    len(sequence),
                ),
            )
            for start in window_starts
        ],
    )

    start_time = time.time()

    for window_index, start in enumerate(
        window_starts
    ):
        end = min(
            start + cfg.ESM2_WINDOW_SIZE,
            len(sequence),
        )

        window_sequence = sequence[
            start:end
        ]

        encoded = tokenizer(
            window_sequence,
            return_tensors="pt",
            add_special_tokens=True,
        )

        encoded = {
            key: value.to(device)
            for key, value
            in encoded.items()
        }

        with torch.inference_mode():
            if (
                device.type == "cuda"
                and cfg.ESM2_USE_FP16
            ):
                with torch.autocast(
                    device_type="cuda",
                    dtype=torch.float16,
                ):
                    outputs = model(
                        **encoded
                    )
            else:
                outputs = model(
                    **encoded
                )

        token_embeddings = (
            outputs.last_hidden_state[
                0,
                1 : 1 + len(window_sequence),
                :,
            ]
            .float()
            .cpu()
            .numpy()
        )

        if (
            token_embeddings.shape[0]
            != len(window_sequence)
        ):
            raise ValueError(
                "Unexpected token/residue alignment: "
                f"{token_embeddings.shape[0]} tokens for "
                f"{len(window_sequence)} residues."
            )

        embedding_sum[
            start:end
        ] += token_embeddings

        embedding_count[
            start:end
        ] += 1

        print(
            f"Window {window_index + 1}/"
            f"{len(window_starts)} complete: "
            f"{start + 1}-{end}"
        )

        del (
            outputs,
            token_embeddings,
            encoded,
        )
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    if embedding_count.min() < 1:
        raise ValueError(
            "Some residues did not receive an embedding."
        )

    embeddings = (
        embedding_sum
        / embedding_count[:, None]
    ).astype(np.float32)

    metadata = {
        "reference_id": reference_id,
        "sequence_length": len(sequence),
        "model_source": model_source,
        "model_name": cfg.ESM2_MODEL_NAME,
        "hidden_size": hidden_size,
        "window_size": (
            cfg.ESM2_WINDOW_SIZE
        ),
        "stride": cfg.ESM2_STRIDE,
        "window_starts_zero_based": (
            window_starts
        ),
        "minimum_window_coverage": int(
            embedding_count.min()
        ),
        "maximum_window_coverage": int(
            embedding_count.max()
        ),
        "runtime_seconds": (
            time.time() - start_time
        ),
    }

    np.save(
        embedding_path,
        embeddings,
    )

    with metadata_path.open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            metadata,
            file,
            indent=2,
        )

    del model, tokenizer
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return embeddings, metadata

## 11. Model 3 — purpose: test whether ESM-2 contextual information improves the biologically informed XGBoost model

In [ ]:
if cfg.RUN_XGB_ESM2:
    if fasta_path is None:
        raise FileNotFoundError(
            "The reference FASTA is required for ESM-2."
        )

    (
        position_embeddings,
        esm_metadata,
    ) = extract_reference_embeddings(
        fasta_path
    )

    if position_embeddings.shape[0] != 1159:
        raise ValueError(
            "Unexpected number of ESM-2 position embeddings."
        )

    zero_based_positions = (
        df["position"].to_numpy(
            dtype=np.int32
        )
        - 1
    )

    variant_embeddings = (
        position_embeddings[
            zero_based_positions
        ]
    )

    base_matrix = (
        df[
            biochemical_domain_features
        ]
        .to_numpy(dtype=np.float32)
    )

    X_esm2 = np.concatenate(
        [
            base_matrix,
            variant_embeddings,
        ],
        axis=1,
    ).astype(np.float32)

    esm_feature_names = [
        f"esm2_{index:03d}"
        for index in range(
            position_embeddings.shape[1]
        )
    ]

    combined_feature_names = (
        biochemical_domain_features
        + esm_feature_names
    )

    (
        esm2_metrics,
        esm2_oof,
        esm2_importance,
    ) = run_xgboost_cv(
        X=X_esm2,
        feature_names=(
            combined_feature_names
        ),
        model_name=(
            "model_3_xgb_domains_esm2"
        ),
    )

    current_metric_rows.extend(
        esm2_metrics.to_dict(
            orient="records"
        )
    )
    current_oof_frames.append(
        esm2_oof
    )

    if not esm2_importance.empty:
        current_importance_frames.append(
            esm2_importance
        )

    display(
        esm2_metrics.loc[
            esm2_metrics[
                "fold"
            ].eq("OOF")
        ]
    )

    del (
        position_embeddings,
        variant_embeddings,
        base_matrix,
        X_esm2,
    )
    gc.collect()

## 12. Combine current and previous runs — purpose: create one final comparison across all available models

In [ ]:
current_metrics_df = pd.DataFrame(
    current_metric_rows
)

current_oof_df = (
    pd.concat(
        current_oof_frames,
        ignore_index=True,
    )
    if current_oof_frames
    else pd.DataFrame()
)

metric_frames = [
    frame
    for frame in [
        previous_metrics_df,
        current_metrics_df,
    ]
    if not frame.empty
]

if not metric_frames:
    raise ValueError(
        "No model was executed and no previous "
        "metrics were loaded."
    )

all_metrics_df = pd.concat(
    metric_frames,
    ignore_index=True,
)

all_metrics_df["fold"] = (
    all_metrics_df["fold"].astype(str)
)

all_metrics_df = (
    all_metrics_df.drop_duplicates(
        subset=[
            "model",
            "fold",
        ],
        keep="last",
    )
    .sort_values(
        [
            "model",
            "fold",
        ]
    )
    .reset_index(drop=True)
)

oof_frames = [
    frame
    for frame in [
        previous_oof_df,
        current_oof_df,
    ]
    if not frame.empty
]

all_oof_df = (
    pd.concat(
        oof_frames,
        ignore_index=True,
    )
    .drop_duplicates(
        subset=[
            "variant_id",
            "model",
        ],
        keep="last",
    )
    .reset_index(drop=True)
    if oof_frames
    else pd.DataFrame()
)

oof_comparison = (
    all_metrics_df.loc[
        all_metrics_df["fold"].eq(
            "OOF"
        )
    ]
    .sort_values(
        "spearman",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(oof_comparison)

## 13. Functional-range and error analysis — purpose: determine where each model succeeds or fails biologically

In [ ]:
category_metric_rows = []

if not all_oof_df.empty:
    for (
        model_name,
        category_name,
    ), group in all_oof_df.groupby(
        [
            "model",
            "functional_category",
        ]
    ):
        if len(group) < 2:
            continue

        category_metrics = (
            calculate_metrics(
                group["y_true"].to_numpy(),
                group["y_pred"].to_numpy(),
            )
        )

        category_metric_rows.append(
            {
                "model": model_name,
                "functional_category": (
                    category_name
                ),
                "n": len(group),
                **category_metrics,
            }
        )

category_metrics_df = pd.DataFrame(
    category_metric_rows
)

if not category_metrics_df.empty:
    display(
        category_metrics_df.sort_values(
            [
                "model",
                "functional_category",
            ]
        )
    )

worst_errors_df = (
    all_oof_df.sort_values(
        "absolute_error",
        ascending=False,
    )
    .groupby(
        "model",
        group_keys=False,
    )
    .head(30)
    .reset_index(drop=True)
    if not all_oof_df.empty
    else pd.DataFrame()
)

if not worst_errors_df.empty:
    display(
        worst_errors_df[
            [
                "model",
                "variant_id",
                "position",
                "y_true",
                "y_pred",
                "absolute_error",
                "functional_category",
            ]
        ].head(50)
    )

## 14. Model-comparison figures — purpose: visualize ranking performance and prediction calibration

In [ ]:
if not oof_comparison.empty:
    plt.figure(figsize=(9, 5))
    plt.bar(
        oof_comparison["model"],
        oof_comparison["spearman"],
    )
    plt.ylabel("OOF Spearman correlation")
    plt.xlabel("Model")
    plt.title(
        "CardioVUS model comparison\\n"
        "Residue-grouped 3-fold validation"
    )
    plt.xticks(
        rotation=25,
        ha="right",
    )
    plt.tight_layout()
    plt.savefig(
        cfg.FIGURES_DIR
        / "model_comparison_spearman.png",
        dpi=160,
        bbox_inches="tight",
    )
    plt.show()

if not all_oof_df.empty:
    for model_name, model_oof in (
        all_oof_df.groupby("model")
    ):
        plt.figure(figsize=(7, 6))
        plt.hexbin(
            model_oof["y_true"],
            model_oof["y_pred"],
            gridsize=45,
            mincnt=1,
        )
        plt.colorbar(
            label="Number of variants"
        )
        plt.xlabel(
            "Observed trafficking score"
        )
        plt.ylabel(
            "OOF predicted trafficking score"
        )
        plt.title(
            f"{model_name}\\n"
            "Observed vs residue-grouped OOF prediction"
        )
        plt.tight_layout()

        safe_name = re.sub(
            r"[^A-Za-z0-9_-]+",
            "_",
            model_name,
        )

        plt.savefig(
            cfg.FIGURES_DIR
            / f"{safe_name}_observed_vs_predicted.png",
            dpi=160,
            bbox_inches="tight",
        )
        plt.show()

## 15. Feature importance — purpose: identify which interpretable variables XGBoost used most strongly

In [ ]:
current_importance_df = (
    pd.concat(
        current_importance_frames,
        ignore_index=True,
    )
    if current_importance_frames
    else pd.DataFrame()
)

if not current_importance_df.empty:
    importance_summary_df = (
        current_importance_df.groupby(
            [
                "model",
                "feature",
            ],
            as_index=False,
        )["gain"]
        .mean()
    )

    importance_summary_df[
        "normalized_gain"
    ] = (
        importance_summary_df["gain"]
        / importance_summary_df.groupby(
            "model"
        )["gain"].transform("sum")
    )

    display(
        importance_summary_df.sort_values(
            [
                "model",
                "normalized_gain",
            ],
            ascending=[
                True,
                False,
            ],
        ).groupby(
            "model",
            group_keys=False,
        ).head(
            cfg.TOP_IMPORTANCE_FEATURES
        )
    )

    for model_name, group in (
        importance_summary_df.groupby(
            "model"
        )
    ):
        top_group = (
            group.sort_values(
                "normalized_gain",
                ascending=False,
            )
            .head(
                cfg.TOP_IMPORTANCE_FEATURES
            )
            .sort_values(
                "normalized_gain"
            )
        )

        plt.figure(
            figsize=(
                10,
                max(
                    6,
                    0.26 * len(top_group),
                ),
            )
        )
        plt.barh(
            top_group["feature"],
            top_group[
                "normalized_gain"
            ],
        )
        plt.xlabel(
            "Mean normalized gain"
        )
        plt.ylabel("Feature")
        plt.title(
            f"{model_name}\\n"
            "Top XGBoost feature importances"
        )
        plt.tight_layout()

        safe_name = re.sub(
            r"[^A-Za-z0-9_-]+",
            "_",
            model_name,
        )

        plt.savefig(
            cfg.FIGURES_DIR
            / f"{safe_name}_feature_importance.png",
            dpi=160,
            bbox_inches="tight",
        )
        plt.show()
else:
    importance_summary_df = (
        pd.DataFrame()
    )

## 16. Save outputs — purpose: preserve all evidence needed for comparison, documentation, and the second Kaggle run

In [ ]:
metrics_path = (
    cfg.METRICS_DIR
    / "model_metrics.csv"
)

oof_path = (
    cfg.PREDICTIONS_DIR
    / "oof_predictions.parquet"
)

category_metrics_path = (
    cfg.METRICS_DIR
    / "functional_category_metrics.csv"
)

worst_errors_path = (
    cfg.PREDICTIONS_DIR
    / "largest_oof_errors.csv"
)

importance_path = (
    cfg.METRICS_DIR
    / "feature_importance_gain.csv"
)

config_path = (
    cfg.OUTPUT_DIR
    / "run_config.json"
)

all_metrics_df.to_csv(
    metrics_path,
    index=False,
)

if not all_oof_df.empty:
    all_oof_df.to_parquet(
        oof_path,
        index=False,
    )

if not category_metrics_df.empty:
    category_metrics_df.to_csv(
        category_metrics_path,
        index=False,
    )

if not worst_errors_df.empty:
    worst_errors_df.to_csv(
        worst_errors_path,
        index=False,
    )

if not importance_summary_df.empty:
    importance_summary_df.to_csv(
        importance_path,
        index=False,
    )

with config_path.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        {
            key: str(value)
            if isinstance(value, Path)
            else value
            for key, value
            in asdict(cfg).items()
        },
        file,
        indent=2,
    )

run_manifest = {
    "dataset_path": str(dataset_path),
    "schema_path": str(schema_path),
    "fasta_path": (
        str(fasta_path)
        if fasta_path is not None
        else None
    ),
    "esm2_model_directory": (
        str(esm2_model_directory)
        if esm2_model_directory
        is not None
        else None
    ),
    "models_in_comparison": (
        oof_comparison[
            "model"
        ].tolist()
    ),
    "best_model_by_spearman": (
        oof_comparison.iloc[0][
            "model"
        ]
        if not oof_comparison.empty
        else None
    ),
    "best_oof_spearman": (
        float(
            oof_comparison.iloc[0][
                "spearman"
            ]
        )
        if not oof_comparison.empty
        else None
    ),
}

with (
    cfg.OUTPUT_DIR
    / "run_manifest.json"
).open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        run_manifest,
        file,
        indent=2,
    )

archive_path = shutil.make_archive(
    base_name=str(
        cfg.WORKING_DIR
        / "cardiovus_outputs"
    ),
    format="zip",
    root_dir=cfg.OUTPUT_DIR,
)

print("Metrics:", metrics_path)
print("OOF predictions:", oof_path)
print("Output archive:", archive_path)

display(oof_comparison)

## 17. Interpretation checklist — purpose: convert model performance into a scientifically defensible conclusion

After both runs, document:

1. Whether Models 1 and 2 outperform the constant baseline.
2. Whether UniProt domain features improve Spearman, MAE, and RMSE.
3. Whether ESM-2 improves generalization to unseen residue positions.
4. Which functional range is hardest to predict.
5. Whether the model systematically regresses zero scores toward the mean.
6. Which biochemical or domain features dominate XGBoost gain.
7. Whether the improvement justifies the added ESM-2 complexity.
8. That the system predicts surface trafficking, not clinical pathogenicity.